# 第 7 章互动实验：立方体的针孔投影

本实验用合成三维点和 OpenCV `projectPoints` 作为可视化基线，不给出 `7-1_针孔投影.py` 的矩阵实现。先调节深度和旋转角，观察近大远小及有效深度；后面三个实验对应教程 7.1、7.2 节的实测，都要用到你自己写的 `project_points`。

In [ ]:
%matplotlib inline
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "PingFang SC", "Heiti SC", "Noto Sans CJK SC", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

K = np.array([[520.0, 0, 320.0], [0, 520.0, 240.0], [0, 0, 1.0]])
dist = np.zeros(5)
cube = np.float32([
    [-1,-1,-1], [1,-1,-1], [1,1,-1], [-1,1,-1],
    [-1,-1, 1], [1,-1, 1], [1,1, 1], [-1,1, 1],
])
edges = [(0,1),(1,2),(2,3),(3,0),(4,5),(5,6),(6,7),(7,4),(0,4),(1,5),(2,6),(3,7)]

In [ ]:
def show_projection(depth=7.0, yaw_deg=25):
    yaw = np.deg2rad(yaw_deg)
    R = np.array([[np.cos(yaw), 0, np.sin(yaw)], [0, 1, 0], [-np.sin(yaw), 0, np.cos(yaw)]], dtype=np.float64)
    rvec, _ = cv2.Rodrigues(R)
    tvec = np.array([0.0, 0.0, depth])
    camera_points = (R @ cube.astype(np.float64).T).T + tvec
    pixels, _ = cv2.projectPoints(cube, rvec, tvec, K, dist)
    pixels = pixels.reshape(-1, 2)
    valid = camera_points[:, 2] > 0
    fig, ax = plt.subplots(figsize=(7, 5.3))
    ax.set_xlim(0, 640); ax.set_ylim(480, 0); ax.set_aspect('equal')
    ax.set_facecolor('#101820')
    for a, b in edges:
        if valid[a] and valid[b]: ax.plot(pixels[[a,b],0], pixels[[a,b],1], color='#57d6ff', linewidth=2)
    ax.scatter(pixels[valid,0], pixels[valid,1], c='#ffcc66', s=35)
    span = np.ptp(pixels[valid], axis=0) if np.any(valid) else np.array([np.nan, np.nan])
    ax.set_title(f'depth={depth:.1f}, yaw={yaw_deg}°，投影跨度 {span[0]:.1f}×{span[1]:.1f} px，有效点 {valid.sum()}/8')
    ax.set_xlabel('u'); ax.set_ylabel('v'); ax.grid(alpha=0.2); plt.show()

interact(
    show_projection,
    depth=FloatSlider(min=1.5, max=14.0, step=0.5, value=7.0),
    yaw_deg=IntSlider(min=-75, max=75, step=5, value=25),
);

## 接上自己的作业

下面的实验有一部分要调用你在 `7-1_针孔投影.py` 里写的 `project_points`。先运行这个单元把作业文件加载进来。作业还没写完也可以往下运行，用到它的单元会打印一行提示，不会报错；写完后回来重新运行这个单元。

每个实验都是先写下预测，再运行，然后把结果和原因记到 `实验记录.md`。

In [ ]:
import importlib.util, os
from pathlib import Path

CHAPTER_NAME = "07_传统CV综合"
ASSIGNMENT = "7-1_针孔投影.py"

def find_chapter(name):
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if p.name == name:
            return p
        if (p / name).is_dir():
            return p / name
    raise FileNotFoundError("请在课程目录内启动 Jupyter")

def load_assignment():
    """加载作业文件；设置了环境变量 COURSE_IMPL_DIR 时改用那里的实现（维护者验证用）。"""
    chapter = find_chapter(CHAPTER_NAME)
    override = os.environ.get("COURSE_IMPL_DIR")
    base = chapter
    if override:
        # 相对路径先按当前目录找，找不到再按课程根目录找
        roots = [Path(override), chapter.parent / override]
        base = next((r / chapter.name for r in roots if (r / chapter.name / ASSIGNMENT).is_file()), roots[0] / chapter.name)
    spec = importlib.util.spec_from_file_location(Path(ASSIGNMENT).stem.replace("-", "_"), base / ASSIGNMENT)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

def attempt(fn, *args, **kwargs):
    """调用作业函数；还没实现时打印提示并返回 None，不中断 Notebook。"""
    try:
        return fn(*args, **kwargs)
    except NotImplementedError as exc:
        print(f"先完成作业里的 {fn.__name__}：{exc}")
        return None

mod = load_assignment()   # 改了作业文件之后，重新运行这个单元再往下做
print("作业文件：", find_chapter(CHAPTER_NAME) / ASSIGNMENT)


## 实验 1：自己的投影和 `cv2.projectPoints` 对照（用到作业 7-1）

先预测：

1. 立方体整个在相机前方时，两者最大相差多少像素？
2. 同一个点 (0.3, 0.2, Z) 分别放在相机前方 5 个单位和后方 5 个单位，你的函数和 OpenCV 各返回什么？

运行后对照教程 7.1 节“实测：约定弄反时是什么样”的最后一段。

In [ ]:
def yaw_rotation(yaw_deg):
    yaw = np.deg2rad(yaw_deg)
    return np.array([[np.cos(yaw), 0, np.sin(yaw)], [0, 1, 0], [-np.sin(yaw), 0, np.cos(yaw)]], dtype=np.float64)

def by_opencv(points, R, t, camera=K):
    rvec, _ = cv2.Rodrigues(R)
    return cv2.projectPoints(np.asarray(points, np.float64), rvec, np.asarray(t, np.float64), camera, dist)[0].reshape(-1, 2)

R_demo, t_demo = yaw_rotation(25), np.array([0.3, -0.2, 7.0])
mine = attempt(mod.project_points, cube, K, R_demo, t_demo)
if mine is not None:
    print(f'立方体 8 个顶点，和 cv2.projectPoints 的最大差：{np.abs(mine - by_opencv(cube, R_demo, t_demo)).max():.2e} px')
    for name, z in [('前方 5 个单位', 5.0), ('后方 5 个单位', -5.0)]:
        point = np.array([[0.3, 0.2, z]])
        print(f'点 (0.3, 0.2, {z:+.0f})，{name}：你的输出 {np.round(mod.project_points(point, K, np.eye(3), np.zeros(3))[0], 1)}，'
              f'OpenCV 的输出 {np.round(by_opencv(point, np.eye(3), np.zeros(3))[0], 1)}，主点 ({K[0, 2]:.0f}, {K[1, 2]:.0f})')

## 实验 2：约定弄反是什么样（用到作业 7-1）

位姿是绕 y 轴转 25 度、平移 (0.3, -0.2, 7)。用同一个 `project_points`，故意按三种错误的方式喂参数：

- 旋转矩阵写成了转置；
- 先平移后旋转，也就是算成 R (P + t)；
- 把 t 当成相机在世界里的位置，也就是算成 R (P - t)。

先预测：三种写错各自会让立方体偏多远？哪一种会让顶点全部落到相机后方、被深度检查拦住？

运行后对照教程 7.1 节的图 1 和它下面的表。教程用的是另一台相机和 300 个随机位姿，数值不会相同，对照的是三种错误各自的症状。

In [ ]:
correct = attempt(mod.project_points, cube, K, R_demo, t_demo)
if correct is not None:
    wrong_ways = [
        ('R 写成了转置', R_demo.T, t_demo),
        ('先平移后旋转', R_demo, R_demo @ t_demo),            # R (P + t) = R P + R t
        ('把 t 当成相机位置', R_demo, -R_demo @ t_demo),      # R (P - t) = R P - R t
    ]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
    for ax, (name, R_used, t_used) in zip(axes, wrong_ways):
        wrong = mod.project_points(cube, K, R_used, t_used)
        valid = ~np.isnan(wrong).any(axis=1)
        if valid.any():
            shift = float(np.median(np.linalg.norm(wrong[valid] - correct[valid], axis=1)))
            print(f'{name}：有效顶点 {int(valid.sum())}/8，顶点偏差的中位数 {shift:.1f} px')
        else:
            print(f'{name}：8 个顶点全部在相机后方，返回的都是 NaN，画不出来')
        for a, b in edges:
            ax.plot(correct[[a, b], 0], correct[[a, b], 1], color='gray', linewidth=1.5)
            if valid[a] and valid[b]:
                ax.plot(wrong[[a, b], 0], wrong[[a, b], 1], color='crimson', linewidth=1.5)
        ax.set_xlim(0, 640); ax.set_ylim(480, 0); ax.set_aspect('equal'); ax.grid(alpha=0.2)
        ax.set_title(name + '（灰：正确，红：写错）')
    plt.tight_layout(); plt.show()

## 实验 3：焦距乘 2、距离也乘 2，画面一样吗（用到作业 7-1）

立方体正对相机（不旋转）。一组用焦距 520、距离 4；另一组把焦距改成 1040，同时把立方体推到 8。

先预测：两幅投影完全重合吗？立方体前面那个面和后面那个面的宽度之比，两组各是多少？

运行后想一想：只看一个平面物体的投影时，焦距和距离能分开吗？对照教程 7.2 节“内参不准时位姿被带偏多少”。

In [ ]:
settings = [('焦距 520，距离 4', 520.0, 4.0, 'tab:blue'), ('焦距 1040，距离 8', 1040.0, 8.0, 'tab:orange')]
first = attempt(mod.project_points, cube, K, np.eye(3), np.array([0.0, 0.0, 4.0]))
if first is not None:
    fig, ax = plt.subplots(figsize=(6.4, 4.8))
    for name, focal, depth, color in settings:
        camera = np.array([[focal, 0, 320.0], [0, focal, 240.0], [0, 0, 1.0]])
        pixels = mod.project_points(cube, camera, np.eye(3), np.array([0.0, 0.0, depth]))
        near_width = pixels[1, 0] - pixels[0, 0]      # 顶点 0、1 在靠近相机的那个面上
        far_width = pixels[5, 0] - pixels[4, 0]       # 顶点 4、5 在远离相机的那个面上
        print(f'{name}：前面的面宽 {near_width:.1f} px，后面的面宽 {far_width:.1f} px，比值 {near_width / far_width:.3f}')
        for a, b in edges:
            ax.plot(pixels[[a, b], 0], pixels[[a, b], 1], color=color, linewidth=1.8, label=name if (a, b) == edges[0] else None)
    ax.set_xlim(0, 640); ax.set_ylim(480, 0); ax.set_aspect('equal'); ax.grid(alpha=0.2); ax.legend(loc='lower right')
    ax.set_title('中间那个平面的大小相同，前后两个面不同')
    plt.show()

## 破坏实验与记录

- 在最上面的互动单元里把深度从 14 降到 2，记录包围盒尺度变化和无效深度。
- 修改 `K` 中焦距或主点，区分缩放与整体平移。
- 实验 1 里把 `t_demo` 的 z 从 7 减到 0.5，让立方体的一部分顶点转到相机后面。先预测你的函数和 OpenCV 各返回什么，再看哪几条棱该画、哪几条不该画。
- 实验 2 里把旋转角从 25 度改成 0 度，“R 写成了转置”那一栏还看得出错吗？为什么验收里要用非对称的旋转？
- 思考同一条相机射线上的两个点为何投到同一像素。

把 `K`、位姿、深度范围、投影跨度与失败原因写入 `实验记录.md`。